In [ ]:
# %% [markdown]
# # LoopNav rollout evaluation analysis
#
# Loads the per-(arm, clip) records collected by `scripts/world/eval_rollout.py`
# (per-step PSNR/LPIPS/DreamSim + alignment covariates + retrieval
# diagnostics) and compares the heuristic baselines (Temporal, WorldMem), the
# LongLive-RAG content-similarity baseline, and the FAR ablation ladder
# (pretrained encoder -> + MLP adapter -> + metadata channel at a fixed gate
# -> + learned gate, i.e. FAR -- Multi-Cue):
#   (1) trajectories binned into SHORT / MIDDLE / LONG (terciles of step
#       count), one comparison figure per bin;
#   (2) within each bin, per-step metrics are aligned by NORMALIZING the
#       trajectory to the unit interval and averaging across clips at
#       corresponding fractional timepoints (bootstrap CIs over clips);
#   (3) additional views: full-trajectory summary table, END-ALIGNED real-time
#       curves (composition-safe loop-closure regime), per-clip PAIRED
#       differences vs the temporal baseline (Wilcoxon), and retrieval
#       diagnostics vs progress.
#
# Needs: a record battery under `results/rollout_eval/<battery>/<arm>/*.npz`
# (one sub-directory per arm plus `protocol.json`; `EVAL_DIR` below names the
# battery). Runs on the CPU.

# %%
import glob
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

from far.constants import ROOT

os.chdir(ROOT)   # EVAL_DIR is repo-root-relative (Jupyter kernels start in notebooks/)

ROUTE = "ABCA"       # "ABA" | "ABCA" | "both" — restricts ALL downstream views
EVAL_DIR = "results/rollout_eval/loopnav_0700000"   # the battery: one record dir per arm

# `pnp_longlive` = LongLive-RAG (arXiv:2606.02553), the action-agnostic
# content-similarity retrieval baseline: query_conditioning=none,
# scoring=visual, frozen offline-trained encoder.
ARMS   = ["B_posefa_temporal", "pnp_longlive", "B_posefa_worldmem",
          "pnp_jepa_frozen", "pnp_visual_only", "pnp_ablz", "pnp_learned_vmz"]
COLORS = dict(zip(ARMS, ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple", "gold",
                         "tab:cyan"]))
# Display names — used for every legend, tick label and summary-table row, so
# the run keys stay the on-disk directory names while the figures read as the
# arm ladder: baselines, then the retriever built up one component at a time.
PRETTY = {
    "B_posefa_temporal": "Temporal",
    "pnp_longlive":      "LongLive-RAG",
    "B_posefa_worldmem": "WorldMem",
    "pnp_jepa_frozen":   "Enc. Pretrained",
    "pnp_visual_only":   "+ MLP Adapter",
    "pnp_ablz":          "+ Meta, lambda=0.5",
    "pnp_learned_vmz":   "+ Meta, lambda learned",
}
pretty = lambda a: PRETTY.get(a, a)
# ARMS   = ["B_posefa_temporal", "B_posefa_worldmem", "pnp_meta_only", "pnp_visual_only", "pnp_learned_vmz"]
# COLORS = dict(zip(ARMS, ["tab:blue", "tab:orange", "tab:cyan", "tab:red", "tab:purple"]))
METRICS = ["psnr", "lpips", "dreamsim"]
BETTER  = {"psnr": "higher is better", "lpips": "lower is better", "dreamsim": "lower is better"}

protocol = json.load(open(f"{EVAL_DIR}/protocol.json"))
STRIDE, FPS = protocol["rollout_stride"], 20.0

recs = {arm: {int(Path(p).stem): dict(np.load(p, allow_pickle=False))
              for p in sorted(glob.glob(f"{EVAL_DIR}/{arm}/*.npz"))}
        for arm in ARMS}
# An arm whose records have not been collected yet would empty the paired
# intersection below and abort the notebook. Drop it with a warning instead, so
# a newly added arm can be wired in here BEFORE its rollout_eval battery
# finishes.
_missing = [a for a in ARMS if not recs[a]]
if _missing:
    print(f"[skip] no records under {EVAL_DIR} for {_missing} -- "
          f"run scripts/world/eval_rollout.py to collect them")
    ARMS = [a for a in ARMS if a not in _missing]
    recs = {a: recs[a] for a in ARMS}
# Paired design: analyse the clips COMPLETED BY EVERY ARM. On a finished
# battery this is all of them; on an in-progress one it is the common subset,
# so the notebook can be re-run while collection continues (results tighten as
# records land — pairing stays exact either way).
clips = sorted(set.intersection(*(set(recs[a]) for a in ARMS)))
assert clips, f"no clips completed by every arm yet under {EVAL_DIR}"
per_arm = {a: len(recs[a]) for a in ARMS}
if any(n != len(clips) for n in per_arm.values()):
    print(f"in-progress battery: per-arm records {per_arm} -> analysing "
          f"{len(clips)} common clips")
for arm in ARMS:
    recs[arm] = {c: recs[arm][c] for c in clips}

meta = {}
for c in clips:
    r0 = recs[ARMS[0]][c]
    parts = Path(str(r0["video_id"])).parts
    n = int(r0["n_steps"])
    assert all(int(recs[a][c]["n_steps"]) == n for a in ARMS)
    meta[c] = dict(route=parts[-4], scene=parts[-3],
                   family=parts[-3].rsplit("_", 1)[0], n_steps=n)
if ROUTE != "both":
    clips = [c for c in clips if meta[c]["route"] == ROUTE]
    assert clips, f"no {ROUTE} clips in the common subset"
    recs = {a: {c: recs[a][c] for c in clips} for a in ARMS}
    meta = {c: meta[c] for c in clips}
    print(f"route filter: analysing {len(clips)} {ROUTE} clips")
print(f"{len(clips)} clips x {len(ARMS)} arms | stride {STRIDE}, "
      f"context_size {protocol['context_size']}, ddim{protocol['sampling_steps']} | "
      f"ABA {sum(m['route'] == 'ABA' for m in meta.values())} / "
      f"ABCA {sum(m['route'] == 'ABCA' for m in meta.values())}")

# %%
# Trajectory-length bins: terciles of the step count, so the three bins are
# equal-sized by construction. Lengths reported in generation steps and in
# seconds of GT trajectory (steps x stride / fps).
steps_all = np.array([meta[c]["n_steps"] for c in clips])
q1, q2 = np.percentile(steps_all, [100 / 3, 200 / 3])
BINS = {
    "short":  [c for c in clips if meta[c]["n_steps"] <= q1],
    "middle": [c for c in clips if q1 < meta[c]["n_steps"] <= q2],
    "long":   [c for c in clips if meta[c]["n_steps"] > q2],
}
for name, cs in BINS.items():
    s = np.array([meta[c]["n_steps"] for c in cs])
    print(f"{name:6s}: {len(cs):3d} clips | steps [{s.min()}..{s.max()}] "
          f"= [{s.min() * STRIDE / FPS:.1f}..{s.max() * STRIDE / FPS:.1f}] s")



In [ ]:
def boot_ci(mat, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    bm = mat[rng.integers(0, mat.shape[0], (B, mat.shape[0]))].mean(axis=1)
    return np.percentile(bm, [2.5, 97.5], axis=0)

# %%
# Full-trajectory summary: mean of per-clip means (each clip weighted equally)
# with 95% bootstrap CIs, per bin and overall.
def summarize(cs, metric, arm):
    v = np.array([recs[arm][c][metric].mean() for c in cs])
    lo, hi = boot_ci(v[:, None])
    return v.mean(), float(lo[0]), float(hi[0])


for metric in METRICS:
    print(f"\n=== {metric} ({BETTER[metric]}) ===")
    header = f"{'arm':24s}" + "".join(f"{b:>22s}" for b in BINS) + f"{'ALL':>22s}"
    print(header)
    for arm in ARMS:
        row = f"{pretty(arm):24s}"
        for cs in list(BINS.values()) + [clips]:
            m, lo, hi = summarize(cs, metric, arm)
            row += f"{m:9.3f} [{lo:.3f},{hi:.3f}]"
        print(row)

# %%
# Box plots over ALL clips — the distribution behind the ALL column above.
# Row 1: per-clip full-trajectory means, one box per arm (box = IQR, line =
# median, x = mean; fliers hidden). Row 2: the SAME means with their 95%
# bootstrap CIs on their own axis — at n = len(clips) the CI is ~1/sqrt(n) of
# the per-clip spread, so it is invisible at the box-plot scale and needs a
# zoomed panel to be compared across arms.
short = pretty          # display name (see PRETTY in the setup cell)
fig, axes = plt.subplots(2, len(METRICS), figsize=(5.2 * len(METRICS), 8),
                         gridspec_kw=dict(height_ratios=[2, 1]), squeeze=False)
for j, metric in enumerate(METRICS):
    per_arm, stats = {}, {}
    for a in ARMS:
        v = np.array([recs[a][c][metric].mean() for c in clips])
        lo, hi = boot_ci(v[:, None])
        per_arm[a], stats[a] = v, (v.mean(), float(lo[0]), float(hi[0]))

    ax = axes[0, j]
    bp = ax.boxplot([per_arm[a] for a in ARMS], patch_artist=True, showfliers=False,
                    medianprops=dict(color="black"))
    for patch, a in zip(bp["boxes"], ARMS):
        patch.set_facecolor(COLORS[a]); patch.set_alpha(0.5)
    ax.plot(range(1, len(ARMS) + 1), [stats[a][0] for a in ARMS], "x", color="black",
            ms=7, mew=1.5, zorder=3, label="mean")
    ax.set_xticks(range(1, len(ARMS) + 1))
    ax.set_xticklabels([short(a) for a in ARMS], fontsize=7, rotation=20, ha="right")
    ax.set_title(f"{metric} per clip ({BETTER[metric]})", fontsize=10)
    ax.grid(alpha=0.3, axis="y")
    if j == 0:
        ax.legend(fontsize=8, loc="best")

    ax = axes[1, j]
    for k, a in enumerate(ARMS):
        m, lo, hi = stats[a]
        ax.errorbar([k], [m], yerr=[[m - lo], [hi - m]], fmt="o", ms=6, lw=2,
                    capsize=4, color=COLORS[a])
    ax.set_xticks(range(len(ARMS)))
    ax.set_xticklabels([short(a) for a in ARMS], fontsize=7, rotation=20, ha="right")
    ax.set_title(f"mean ± 95% bootstrap CI  (n={len(clips)} clips)", fontsize=9)
    ax.grid(alpha=0.3, axis="y")
plt.suptitle("Full-trajectory metrics aggregated over all clips", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# %%
# NEAR-LOOP-CLOSURE tables — the paper's main comparison.
#
# The full-trajectory mean averages the regime where memory is IRRELEVANT (early
# steps: the current frame still predicts the next) together with the regime the
# method exists for (the approach to loop closure, where the answer must come
# from the pool). That dilution can be severe: the ladder's gains concentrate
# in the last quarter of the trajectory, and a difference that is clearly
# positive in the closure window can average out to nothing (or flip sign)
# over the full trajectory.
#
# Two windows are reported, and they should agree:
#   FINAL <CLOSURE_FRAC> OF THE TRAJECTORY — normalized, uses every clip, so
#       clip length cannot bias the comparison. This is the headline table.
#   LAST <END_ALIGN_SEC> s BEFORE CLOSURE — a fixed PHYSICAL window, end-aligned.
#       Drops clips shorter than the window, so absolute numbers sit lower; it is
#       the robustness check, not the headline.
# Every number is the mean of per-clip means (each clip weighted equally) with
# the half-width of a 95% bootstrap CI. The design is PAIRED — same clips, same
# seeds, all arms — so the ladder is compared with a Wilcoxon signed-rank test
# and a per-clip win rate, which is far more powerful than the marginal CIs
# (those can overlap even where the paired difference is highly significant).
CLOSURE_FRAC  = 0.75      # "near closure" = the final (1 - CLOSURE_FRAC) of the trajectory
END_ALIGN_SEC = 5.0       # end-aligned window, in seconds of GT trajectory
LADDER = [("pnp_jepa_frozen", "pnp_visual_only"),        # + MLP adapter
          ("pnp_visual_only", "pnp_ablz"),               # + metadata channel
          ("pnp_ablz", "pnp_learned_vmz"),               # + learned gate
          ("B_posefa_worldmem", "pnp_learned_vmz")]      # vs the strongest baseline
# Which arms belong under the FAR heading. Explicit, NOT a name prefix:
# LongLive-RAG is a pnp_cl variant but a BASELINE, so a prefix rule mislabels it.
OURS = ["pnp_jepa_frozen", "pnp_visual_only", "pnp_ablz", "pnp_learned_vmz"]
OURS_HEADING = "FAR with Vision Encoder"
EMIT_LATEX = True

from scipy.stats import wilcoxon
_U = np.linspace(0.02, 1.0, 25)
_K = max(1, int(round(END_ALIGN_SEC * FPS / STRIDE)))
_elig = [c for c in clips if len(recs[ARMS[0]][c]["psnr"]) >= _K]

def _full(arm, m):    return np.array([recs[arm][c][m].mean() for c in clips])
def _closure(arm, m): return np.array([np.interp(_U, recs[arm][c]["frac"], recs[arm][c][m])[_U >= CLOSURE_FRAC].mean()
                                       for c in clips])
def _endalign(arm, m):return np.array([recs[arm][c][m][-_K:].mean() for c in _elig])

WINDOWS = [(f"FULL TRAJECTORY", _full, clips),
           (f"FINAL {1 - CLOSURE_FRAC:.0%} OF TRAJECTORY  (headline)", _closure, clips),
           (f"LAST {END_ALIGN_SEC:g} s BEFORE CLOSURE  (end-aligned)", _endalign, _elig)]

print(f"route={ROUTE} | {len(clips)} paired clips | stride {STRIDE} @ {FPS:g} fps = "
      f"{STRIDE / FPS:.2f} s/step | end-aligned window {_K} steps, {len(_elig)} clips long enough")
if ROUTE != "both":
    print(f"  NOTE: ROUTE={ROUTE} restricts every number below to that subset — "
          f"set ROUTE='both' in the setup cell for the whole battery.")
print()
_tab = {}
for title, fn, cs in WINDOWS:
    print(f"=== {title}   n={len(cs)}")
    print(f"  {'Retrieval':24s}" + "".join(f"{m:>18s}" for m in METRICS))
    for arm in ARMS:
        if arm in OURS and (ARMS.index(arm) == 0 or ARMS[ARMS.index(arm) - 1] not in OURS):
            print(f"  {'-' * 24}{'-' * 18 * len(METRICS)}")
        row = f"  {pretty(arm):24s}"
        for m in METRICS:
            v = fn(arm, m); lo, hi = boot_ci(v[:, None])
            _tab[(title, arm, m)] = (v.mean(), (float(hi[0]) - float(lo[0])) / 2)
            row += f"{v.mean():11.3f} ±{(float(hi[0]) - float(lo[0])) / 2:.3f}"
        print(row)
    print()

print(f"=== paired ladder in the closure window (final {1 - CLOSURE_FRAC:.0%})")
for a, b in LADDER:
    if a not in ARMS or b not in ARMS:
        continue
    line = f"  {pretty(b):22s} vs {pretty(a):22s}"
    for m in METRICS:
        x, y = _closure(a, m), _closure(b, m)
        d = y - x if BETTER[m].startswith("higher") else x - y      # d > 0 => b better
        line += (f" | {m} {d.mean():+.4f} (p={wilcoxon(d, alternative='greater').pvalue:.0e},"
                 f" win {np.mean(d > 0):.0%})")
    print(line)

# ── LaTeX for the headline table ─────────────────────────────────────────────
if EMIT_LATEX:
    title = WINDOWS[1][0]
    arrow = {"psnr": r"$\uparrow$", "lpips": r"$\downarrow$", "dreamsim": r"$\downarrow$"}
    best = {m: (max if BETTER[m].startswith("higher") else min)(
        _tab[(title, a, m)][0] for a in ARMS) for m in METRICS}
    print("\n" + "% " + "-" * 68 + "\n% headline table, paste into the paper\n\\begin{tabular}{lccc}")
    print("\\toprule")
    print("Retrieval & " + " & ".join(f"{m.upper() if m != 'dreamsim' else 'DreamSim'}{arrow[m]}"
                                      for m in METRICS) + r" \\")
    print("\\midrule")
    for k, arm in enumerate(ARMS):
        if arm in OURS and (k == 0 or ARMS[k - 1] not in OURS):
            print("\\midrule\n\\multicolumn{4}{l}{\\textbf{" + OURS_HEADING + "}} \\\\")
        cells = []
        for m in METRICS:
            v, _ = _tab[(title, arm, m)]
            cells.append(f"\\textbf{{{v:.3f}}}" if v == best[m] else f"{v:.3f}")
        print(f"{pretty(arm).replace('lambda', r'$\lambda$')} & " + " & ".join(cells) + r" \\")
    print("\\bottomrule\n\\end{tabular}")


In [ ]:
# %%
# (1)+(2) Per-bin comparison at NORMALIZED trajectory time: each clip's
# per-step metric is interpolated onto a shared unit-interval grid (using the
# stored `frac` covariate) and averaged across the bin's clips at each
# fractional timepoint; bands are 95% bootstrap CIs over clips (clips are the
# resampling unit — the paired design keeps arms comparable pointwise).
U = np.linspace(0.02, 1.0, 25)


def frac_curves(arm, cs, metric):
    return np.stack([np.interp(U, recs[arm][c]["frac"], recs[arm][c][metric]) for c in cs])


for bin_name, cs in BINS.items():
    fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
    for ax, metric in zip(axes, METRICS):
        for arm in ARMS:
            mat = frac_curves(arm, cs, metric)
            mean = mat.mean(axis=0)
            lo, hi = boot_ci(mat)
            ax.plot(U, mean, color=COLORS[arm], lw=1.8,
                    label=f"{pretty(arm)} (mean {np.mean([recs[arm][c][metric].mean() for c in cs]):.3f})")
            ax.fill_between(U, lo, hi, color=COLORS[arm], alpha=0.15, lw=0)
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
        ax.set_title(f"{metric} vs normalized trajectory time ({BETTER[metric]})", fontsize=10)
    axes[-1].set_xlabel("fraction of trajectory (0 = rollout start, 1 = loop closure)")
    plt.suptitle(f"{bin_name.upper()} trajectories ({len(cs)} clips)", fontsize=13)
    plt.tight_layout()
    plt.show()


In [ ]:
# %%
# BETTER VIEW #1 — END-ALIGNED, REAL-TIME curves. Normalizing to the unit
# interval warps TIME: "progress 0.9" is ~1 s from closure on a short clip but
# ~10 s on a long one, yet drift accumulates in real seconds — so fractional
# averaging mixes different drift depths at every x. Aligning on steps BEFORE
# CLOSURE keeps the physics: every included clip contributes the same real-time
# window, and the loop-closure regime (where the retrieval mechanisms differ
# most) is measured composition-safely.
K = 25                                     # last K steps = K*stride/fps seconds
elig = [c for c in clips if meta[c]["n_steps"] >= K]
x = -np.arange(K)[::-1] * STRIDE / FPS     # seconds before closure (<= 0)
fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
for ax, metric in zip(axes, METRICS):
    for arm in ARMS:
        mat = np.stack([recs[arm][c][metric][-K:] for c in elig])
        mean, (lo, hi) = mat.mean(axis=0), boot_ci(mat)
        ax.plot(x, mean, color=COLORS[arm], lw=1.8, label=pretty(arm))
        ax.fill_between(x, lo, hi, color=COLORS[arm], alpha=0.15, lw=0)
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[-1].set_xlabel(f"seconds before loop closure ({len(elig)} clips with >= {K} steps)")
plt.suptitle("End-aligned (real-time) — the loop-closure approach", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# %%
# BETTER VIEW #2 — PAIRED per-clip differences vs the temporal baseline.
# Bin/arm means hide the pairing power of the design: every arm ran the SAME
# episodes with the SAME noise, so per-clip deltas cancel clip difficulty.
# Boxes show the per-clip distribution of (arm - temporal) full-trajectory
# means; annotations give the two-sided Wilcoxon signed-rank p-value.
BASE = "B_posefa_temporal"
others = [a for a in ARMS if a != BASE]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric in zip(axes, METRICS):
    data, labels, colors = [], [], []
    for bi, (bin_name, cs) in enumerate(BINS.items()):
        base = np.array([recs[BASE][c][metric].mean() for c in cs])
        for arm in others:
            d = np.array([recs[arm][c][metric].mean() for c in cs]) - base
            data.append(d)
            labels.append(f"{bin_name}\n{pretty(arm)}")
            colors.append(COLORS[arm])
    bp = ax.boxplot(data, patch_artist=True, showfliers=False)
    for patch, col in zip(bp["boxes"], colors):
        patch.set_facecolor(col)
        patch.set_alpha(0.5)
    for i, d in enumerate(data):
        p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
        stars = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
        ax.text(i + 1, ax.get_ylim()[1], stars, ha="center", fontsize=11)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, fontsize=6, rotation=90)
    ax.set_title(f"Δ{metric} vs temporal ({BETTER[metric]})", fontsize=10)
    ax.grid(alpha=0.3, axis="y")
plt.suptitle("Paired per-clip differences vs temporal (Wilcoxon: * p<.05 ** p<.01 *** p<.001)")
plt.tight_layout()
plt.show()


In [ ]:
# %%
# Mechanism panel — retrieval diagnostics vs normalized progress: the mean
# metric distance of retrieved frames to the current pose, and the mean |Δt|
# of the picks. Divergent retrieval behaviour here EXPLAINS metric gaps above
# (e.g. geometry-anchored picks staying near the loop-closure region while a
# corrupted visual query drifts).
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (key, name) in zip(axes, [("ret_dist", "mean pick distance to current pose (m)"),
                                  ("ret_dt", "mean pick |Δt| (s)")]):
    for arm in ARMS:
        mats = []
        for c in clips:
            r = recs[arm][c]
            if key not in r:
                continue
            v = np.abs(r[key]).mean(axis=1)
            mats.append(np.interp(U, r["frac"], v))
        if not mats:
            continue
        mat = np.stack(mats)
        mean, (lo, hi) = mat.mean(axis=0), boot_ci(mat)
        ax.plot(U, mean, color=COLORS[arm], lw=1.8, label=pretty(arm))
        ax.fill_between(U, lo, hi, color=COLORS[arm], alpha=0.15, lw=0)
    ax.set_xlabel("fraction of trajectory")
    ax.set_ylabel(name)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.suptitle("Retrieval diagnostics vs progress", fontsize=13)
plt.tight_layout()
plt.show()

# %% [markdown]
# Further views the records support without recollection: distance-to-closure
# alignment (`dist_to_closure` covariate — metrics vs METERS from the closure
# point, the axis the memory actually indexes) and per-scene-family panels
# (parse `family` from video_id — 6 Minecraft biomes).
